# CampusVoice-RT: 재현 노트북 (Colab, NVIDIA L4)

런타임 → GPU 유형을 **L4**로 설정한 뒤 위에서부터 실행. 모든 베이스라인(B1~B4)을 같은 런타임·같은 VAD 설정으로 측정한다.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/ekgP525/asr-challenge.git
%cd asr-challenge
!pip install -q -r requirements.txt

## 평가셋 다운로드 (Hugging Face Datasets, 링크 TBD)

In [ ]:
# TODO: HF dataset id 확정 후 채움
# !huggingface-cli download <dataset-id> --repo-type dataset --local-dir data/eval

## 베이스라인 실행 → 채점

In [ ]:
import subprocess
for sid in ['B1','B2','B3','B4']:
    subprocess.run(['python','baseline/run_qwen3_asr.py','--config',f'baseline/configs/{sid}.json',
                    '--manifest','data/eval/manifest.jsonl','--audio-root','data/eval','--out-dir',f'runs/{sid}'], check=True)
    subprocess.run(['python','evaluation/evaluate.py','--config',f'baseline/configs/{sid}.json',
                    '--manifest','data/eval/manifest.jsonl','--predictions',f'runs/{sid}/predictions.jsonl',
                    '--latency',f'runs/{sid}/latency.jsonl','--env',f'runs/{sid}/environment.json',
                    '--hardware','NVIDIA L4'], check=True)

## 리더보드

In [ ]:
!python evaluation/make_leaderboard.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/leaderboard.md').read()))
display(Image('results/latency_cer_paths.png'))